# Staff Churn Prediction
Machine Learning (ML) Zoomcamp Mid-term Project

## The problem domain

Ask yourself, were it possible:

1. Would you like forewarning of a staff member leaving?
1. Which traits / indicators are particularly conducive to forewarning of staff churn?
1. Could staff survey responses be analysed to calculate a probability of a staff member leaving, so their manager can proactively respond?
1. Could all staff survey responses be regularly analysed to allow HR to plan around forecast churn?

Using a [public dataset](https://www.kaggle.com/datasets/mahmoudemadabdallah/hr-analytics-employee-attrition-and-performance) of employee survey responses and HR data -- for a particular US company and their survey questions -- this project attempts to find whether this is indeed possible, and the use cases that could thus be demonstrated thereby.

The value of such an innovation would be measured in both the attraction, retention, management intervention, and bottom line of such an organisation. However, doing so is not without trade-off.  This is rather the proof-of-concept, and it remains for others to address the operational, compliance and support impacts of such an approach.

After all, people are not data.  But data, here, could perhaps predict the behaviour of one's staff. And that alone, is the art-of-the-possible intended here.

## Objectives

1.  Consider traits and insights that can be gleaned from the dataset to guide / understand the impacts on employees.
1.  Targeting the `Attrition` feature, build a machine learning model to predict whether a given employee is likely to churn
1.  Rank features that promote or reduce the likelihood of churn
1.  "Operationalise" the machine learning model by wrapping it in an API, itself wrapped into a Docker container

## Imports

In [145]:
from IPython.display import display

import numpy as np
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score #, root_mean_squared_error #, mutual_info_score, roc_curve, auc
from sklearn.model_selection import train_test_split #, KFold
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text #, plot_tree, export_graphviz
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from typing import Iterable, TypeVar

skPredict = TypeVar("skPredict", LogisticRegression, DecisionTreeClassifier, RandomForestClassifier)
skDecide = TypeVar("skDecide", skPredict, LinearRegression, DecisionTreeRegressor, RandomForestRegressor)
skFit = TypeVar("skFit", skPredict, skDecide)

# import matplotlib.pyplot as plt
# import seaborn as sns
# %matplotlib inline

# %pip install tqdm
# from tqdm.auto import tqdm

# %pip install XGBoost
# import xgboost as xgb

## Utilities

In [146]:
# https://stackoverflow.com/questions/1175208/elegant-python-function-to-convert-camelcase-to-snake-case
import re

def to_snake_case(name):
    name = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub('([a-z0-9])([A-Z])', r'\1_\2', name)
    return name.replace(' ','_').replace('__','_').replace('(', '').replace(')', '').lower()

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.map(to_snake_case)
    return df

# print(to_snake_case('camel2_camel2_case'))   # camel2_camel2_case
# print(to_snake_case('camel2C Camel2_Case'))  # camel2_camel2_case
# print(to_snake_case('getHTTPResponseCode'))  # get_http_response_code
# print(to_snake_case('HTTPResponseCodeXYZ'))  # http_response_code_xyz

In [147]:
def scalar_features(df: pd.DataFrame):
    cols = df.select_dtypes(exclude=[object, 'category']).columns
    return list(cols)

def categorical_features(df: pd.DataFrame):
    cols = df.select_dtypes(include=[object, 'category']).columns
    return list(cols)

In [148]:
def get_lookup_value(id: int, df_lookup: pd.DataFrame, col:str = 'level', ifNull:str = 'Not Answered') -> str:
    return df_lookup.loc[id][col] if id in df_lookup.index else ifNull

In [149]:
def validation_testing_training_full_split(dataframe: pd.DataFrame, seed: int = 42, validation: float = 0.2, testing: float = 0.2):
    assert 0 < validation and 0 < testing and 1 > (validation + testing)

    validation_of_full = validation / (1 - testing)
    if validation_of_full == 0:
        validation_of_full = None
        
    df_full,     df_testing    = train_test_split(dataframe, test_size=testing,            random_state=seed, shuffle=True)
    df_training, df_validation = train_test_split(df_full,   test_size=validation_of_full, random_state=seed, shuffle=True)
    
    df_validation = df_validation.reset_index(drop=True)
    df_testing = df_testing.reset_index(drop=True)
    df_training = df_training.reset_index(drop=True)
    df_full = df_full.reset_index(drop=True)
    
    return df_validation, df_testing, df_training, df_full

In [150]:
def y_split(dataframe: pd.DataFrame, yColumn: str, drop: Iterable[str] = []):
    columns = set(dataframe.columns)   
    assert columns.issuperset([yColumn]), f'{yColumn} not found in dataframe'
    assert columns.issuperset(drop), f'At least one of {drop} not found in dataframe'
    
    df = dataframe.copy()
    y = df[yColumn]
    for col in drop + [yColumn]:
        del df[col]
        
    return df, y

In [151]:
def regularize(X, r=0.000000001):
    return X + np.eye(X.shape[0]) * r

In [152]:
# def regularize(X, r: float = 0.00000001):
#     return X if r == 1 else X + np.eye(X.shape[0]) * r
#     
# X = [
#     [1, 2, 2],
#     [2, 1, 1], # r
#     [2, 1, 1], # r
#     [0, 9, 9],
#     [1, 0, 0],
# ] #     c  c
# X = np.array(X)
# XTX = X.T.dot(X)
# print(XTX) # duplicate columns is an issue for linear / logistic regression
# np.linalg.inv(XTX)
# 
# regularize(X, r=0.01)

In [153]:
def sigmoid(score):
    return 1 / (1 + np.exp(-score))

# z = np.linspace(-5, 5, 51)
# plt.plot(z, sigmoid(z))

In [154]:
def display_predictive_features_for_target(df: pd.DataFrame, target: str, categorical: Iterable[str] = []):
    global_target = df[target].mean()
    for c in categorical:
        df_group = df.groupby(c)[target].agg('mean','count')
        df_group['diff'] = df_group.mean - global_target
        df_group['risk'] = df_group.mean / global_target
        display(df_group)

In [155]:
def one_hot_encode(df: pd.DataFrame, dv: DictVectorizer = DictVectorizer(sparse=False), drop: Iterable[str] = [], fit: bool = False):
    assert set(df.columns).issuperset(drop), f'At least one of {drop} is not found in the DataFrame `df`'
    
    df_encode = df.copy()
    for feature in drop:
        del df_encode[feature]
    
    data = df_encode.to_dict(orient='records')
    X = dv.fit_transform(data) if fit else dv.transform(data)
        
    assert len(dv.feature_names_) == X.shape[1]
    return X, dv

In [156]:
def fit(model: skFit, df: pd.DataFrame, y: pd.Series, dv: DictVectorizer = DictVectorizer(sparse=False), drop: Iterable[str] = []) -> tuple[skFit, DictVectorizer]:
    assert df.shape[0] == y.shape[0], '`df` and `y` mismatch'
    
    X, dv = one_hot_encode(df, dv, drop, fit=True)
    model.fit(X, y)
    
    return model, dv

In [157]:
def decide(model: skDecide, dv: DictVectorizer, df: pd.DataFrame, drop: Iterable[str] = []):
    X, _ = one_hot_encode(df, dv, drop)
    y_pred = model.predict(X)
    
    return y_pred

In [158]:
def predict(model: skPredict, dv: DictVectorizer, df: pd.DataFrame, drop: Iterable[str] = []):
    X, _ = one_hot_encode(df, dv, drop)
    y_pred = model.predict_proba(X)[:, 1]
    
    return y_pred

In [159]:
def random_predictions(y: pd.Series, seed: int = 42):
    np.random.seed(seed)
    if y.dtype == 'bool':
        return np.random.uniform(0, 1, size=len(y))
    else:
        return np.random.uniform(0, 1, size=len(y))

In [160]:
def model_metrics(y: pd.Series, y_pred: pd.Series, threshold: float = 0.5):
    assert len(y) == len(y_pred), "`y` and `y_pred` mismatch"
    assert 0 <= threshold and threshold <= 1, "invalid threshold"
    
    actual_positive = (y == 1)
    actual_negative = (y == 0)
    predicted_positive = (y_pred >= threshold)
    predicted_negative = (y_pred < threshold)
    
    true_positive = (predicted_positive & actual_positive).sum()
    false_positive = (predicted_positive & actual_negative).sum()
    false_negative = (predicted_negative & actual_positive).sum()
    true_negative = (predicted_negative & actual_negative).sum()
    
    # Accuracy = ratio of correct predictions -- false and positive 
    # accuracy = proportional_matrix[0,0] + proportional_matrix[1,1]
    accuracy = (true_positive + true_negative) / (true_positive + false_positive + false_negative + true_negative)
    
    # Precision = ratio of correct to positive predictions; higher is better
    precision = true_positive / (true_positive + false_positive)
    # Recall = ratio of correctly predicted positive; higher is better
    recall = true_positive / (true_positive + false_negative)
    f1 = 2 * (precision * recall) / (precision + recall)
    
    # TPR = ratio of true positives in all positives; higher is better
    # true_positive_rate = recall
    true_positive_rate  = true_positive / ( false_negative + true_positive)
    # FPR = ratio of false positives in all negatives; lower is better
    false_positive_rate = false_positive / (true_negative + false_positive)
    
    matrix = np.array([
        #   g(Xi) < t    |   g(Xi) >= t
        [ true_negative  , false_positive ], # y == 0
        [ false_negative , true_positive  ]  # y == 1
    ])
    
    proportional_matrix = (matrix / matrix.sum()).round(6)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': true_positive,
        'fp': false_positive,
        'fn': false_negative,
        'tn': true_negative,
        'tpr': true_positive_rate,
        'fpr': false_positive_rate,
        'confusion': matrix,
        'proportional_confusion': proportional_matrix
    }

In [161]:
def model_metrics_for_thresholds(y: pd.Series, y_pred: pd.Series, thresholds: Iterable[float]) -> pd.DataFrame:
    results = []
    for threshold in thresholds:
        metrics = model_metrics(y, y_pred, threshold)
        metrics['threshold'] = threshold
        results.append(metrics)
        
    df_metrics = pd.DataFrame(results)
    df_metrics.set_index('threshold', inplace=True)
    
    return df_metrics

## Data Preparation

### Performance Ratings / `dfPerformance`:

PerformanceID
:   Unique identifier for each performance review.

EmployeeID
:   Unique identifier for the employee being reviewed.

ReviewDate
:   The date of the performance review.

EnvironmentSatisfaction
:   Rating of the employee's satisfaction with their work environment.

JobSatisfaction
:   Rating of the employee's satisfaction with their job.

RelationshipSatisfaction
:   Rating of the employee's satisfaction with workplace relationships.

TrainingOpportunitiesWithinYear
:   Number of training opportunities available to the employee within the year.

TrainingOpportunitiesTaken
:   Number of training opportunities the employee has taken.

WorkLifeBalance
:   Rating of the employee's work-life balance.

SelfRating
:   The employee's self-assessment rating.

ManagerRating
:   The manager's rating of the employee's performance.


### Employees / `dfEmployee`:

EmployeeID
:   Unique identifier for each employee.

FirstName
:   The first name of the employee.

LastName
:   The last name of the employee.

Gender
:   The gender of the employee.

Age
:   The age of the employee.

BusinessTravel
:   The frequency of business travel for the employee.

Department
:   The department in which the employee works.

DistanceFromHome (KM)
:   The distance between the employee's home and workplace in kilometers.

State
:   The state in which the employee resides.

Ethnicity
:   The ethnicity of the employee.

MaritalStatus
:   The marital status of the employee.

Salary
:   The annual salary of the employee.

StockOptionLevel
:   The level of stock options granted to the employee.

OverTime
:   Whether the employee works overtime (Yes/No).

HireDate
:   The date the employee was hired.

Attrition
:   Whether the employee has left the company (Yes/No).

YearsAtCompany
:   The number of years the employee has been with the company.

YearsInMostRecentRole
:   The number of years the employee has been in their most recent role.

YearsSinceLastPromotion
:   The number of years since the employee's last promotion.

YearsWithCurrManager
:   The number of years the employee has worked with their current manager.

### Load Datasets

In [162]:
# Load look-up values

[dfEducation, dfRating, dfSatisfied] = map(standardize_columns, [
    pd.read_csv('./data/EducationLevel.csv', names=['id', 'level'], index_col='id', skiprows=1), 
    pd.read_csv('./data/RatingLevel.csv', names=['id', 'level'], index_col='id', skiprows=1),
    pd.read_csv('./data/SatisfiedLevel.csv', names=['id', 'level'], index_col='id', skiprows=1)
    ])


display(dfEducation)
display(dfRating)
display(dfSatisfied)


# Load source data

[dfEmployee, dfPerformance] = map(standardize_columns, [
    pd.read_csv('./data/Employee.csv'), 
    pd.read_csv('./data/PerformanceRating.csv')
    ])

display( dfPerformance.describe(include='all').T )
display( dfEmployee.describe(include='all').T )

,level
id,
1,No Formal Qualifications
2,High School
3,Bachelors
4,Masters
5,Doctorate


,level
id,
1,Unacceptable
2,Needs Improvement
3,Meets Expectation
4,Exceeds Expectation
5,Above and Beyond


,level
id,
1,Very Dissatisfied
2,Dissatisfied
3,Neutral
4,Satisfied
5,Very Satisfied


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
performance_id,6709,6709,PR01,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employee_id,6709,1280,79F7-78EC,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_date,6709,2771,5/22/2022,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
environment_satisfaction,6709.0,NaN,NaN,NaN,3.872559,0.940701,1.0,3.0,4.0,5.0,5.0
job_satisfaction,6709.0,NaN,NaN,NaN,3.430616,1.152565,1.0,2.0,3.0,4.0,5.0
relationship_satisfaction,6709.0,NaN,NaN,NaN,3.427336,1.156753,1.0,2.0,3.0,4.0,5.0
training_opportunities_within_year,6709.0,NaN,NaN,NaN,2.012968,0.82031,1.0,1.0,2.0,3.0,3.0
training_opportunities_taken,6709.0,NaN,NaN,NaN,1.01729,0.950316,0.0,0.0,1.0,2.0,3.0
work_life_balance,6709.0,NaN,NaN,NaN,3.414667,1.143961,1.0,2.0,3.0,4.0,5.0
self_rating,6709.0,NaN,NaN,NaN,3.984051,0.816432,3.0,3.0,4.0,5.0,5.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
employee_id,1470,1470,3012-1A41,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
first_name,1470,1334,Murdock,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last_name,1470,1441,Ponten,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,1470,4,Female,675,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,1470.0,NaN,NaN,NaN,28.989796,7.993055,18.0,23.0,26.0,34.0,51.0
business_travel,1470,3,Some Travel,1043,NaN,NaN,NaN,NaN,NaN,NaN,NaN
department,1470,3,Technology,961,NaN,NaN,NaN,NaN,NaN,NaN,NaN
distance_from_home_km,1470.0,NaN,NaN,NaN,22.502721,12.811124,1.0,12.0,22.0,33.0,45.0
state,1470,3,CA,875,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ethnicity,1470,7,White,860,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Inspect datasets for preparation tasks & ideas

Glancing at the data, the following needs attention:

1. Performance data:

- [x] what to do if most-recent scores have empty values?  n/a
- [ ] `training`, when offered appears to be readily taken.  Should drive inverse correlation to `attrition`; check
- [x] `self_rating` and `manager_rating` scores never use full 1..5 range.  Would consider standardization, but int values exclude usefulness

2. Join performance to employee data frame using `employee_id`:

- [x] performance data for non-existing employees? drop records
- [x] retain only latest performance data per employee (as prior events may have triggered attrition, but employee left in last year only)
- [x] drop perf data frame

3. Employee data:

- [x] what to do if empty values?  n/a
- [x] what to do if employee has no performance data? consider `attrition`, drop?
- [x] categorical features (`overtime` & `attrition`): convert to binary
- [ ] demographic features (`first_name`, `last_name`, and perhaps `age`, `ethnicity` and `gender` if not significant to models) should be dropped
- [ ] `years_since_promotion` & `years_at_company` -> long, whereas `years_in_role` & `years_with_manager` -> short; "stickiness" should correlate to `attrition`; check
- [x] once merged, `employee_id` should replace the index -> excluded from models
- [x] once merged, compare and perhaps generate features from the `hire_date`, `review_date`, and `years_*` features
- [ ] consider correlation of `business_travel` & `job_role` to `attrition`
- [ ] only 3 `states` and 3 `departments`: can't generalise to USA. Underscores specific to this company!

4. Look-up values:

- [x] look-up values should replace references (for future one-hot encoding, else references will incorrectly be interpreted as scalars)
- [x] drop look-up data frames
- [x] `review_date` and `hire_date` str -> Date convert, and expand to 'yyyy-MM' for trend analysis

### Performance

In [163]:
dfPerformance.isnull().sum()

performance_id                        0
employee_id                           0
review_date                           0
environment_satisfaction              0
job_satisfaction                      0
relationship_satisfaction             0
training_opportunities_within_year    0
training_opportunities_taken          0
work_life_balance                     0
self_rating                           0
manager_rating                        0
dtype: int64

In [164]:
dfPerformance.employee_id.isin(dfEmployee.employee_id).unique()

array([ True])

Observations:

1.  No empty values
1.  Every `employee_id` is found in the `dfEmployee` frame, so no data to drop

### Employee

In [165]:
dfEmployee.isnull().sum()

employee_id                   0
first_name                    0
last_name                     0
gender                        0
age                           0
business_travel               0
department                    0
distance_from_home_km         0
state                         0
ethnicity                     0
education                     0
education_field               0
job_role                      0
marital_status                0
salary                        0
stock_option_level            0
over_time                     0
hire_date                     0
attrition                     0
years_at_company              0
years_in_most_recent_role     0
years_since_last_promotion    0
years_with_curr_manager       0
dtype: int64

In [166]:
dfEmployee[dfEmployee.employee_id.duplicated()]

,employee_id,first_name,last_name,gender,age,business_travel,department,distance_from_home_km,state,ethnicity,...,marital_status,salary,stock_option_level,over_time,hire_date,attrition,years_at_company,years_in_most_recent_role,years_since_last_promotion,years_with_curr_manager


In [167]:
dfEmployee[dfEmployee.employee_id.isin(dfPerformance.employee_id) == False].attrition.value_counts()

attrition
No    190
Name: count, dtype: int64

Observations:

1. No empty values
1. No duplicate `employee_id`s
1. There are 190 employees that have no performance data. 
1. But as none have left, they can be assumed to be satisfied.
1. No data interventions (eg dropping records) necessary.

### Join most-recent Performance data per `employee_id` to Employee data

In [168]:
dfPerformance.review_date = pd.to_datetime(dfPerformance.review_date)  # USA dates format is default

In [169]:
dup_perf_employee_id = dfPerformance[dfPerformance.employee_id.duplicated() == True].tail(1).employee_id.values
display(f'BEFORE: Duplicate performance records for {dup_perf_employee_id[0]}:', dfPerformance[dfPerformance.employee_id.isin(dup_perf_employee_id)])

'BEFORE: Duplicate performance records for 4500-37EB:'

,performance_id,employee_id,review_date,environment_satisfaction,job_satisfaction,relationship_satisfaction,training_opportunities_within_year,training_opportunities_taken,work_life_balance,self_rating,manager_rating
615,PR1545,4500-37EB,2017-03-16,1,3,1,2,1,3,3,2
1331,PR2190,4500-37EB,2018-03-16,3,4,5,1,0,4,4,3
1697,PR252,4500-37EB,2014-03-17,3,2,2,3,0,3,4,3
2150,PR2928,4500-37EB,2019-03-16,4,5,2,2,1,5,5,4
3104,PR3788,4500-37EB,2020-03-15,5,4,5,1,0,3,4,3
4195,PR4770,4500-37EB,2021-03-15,4,4,4,2,1,5,4,3
5182,PR566,4500-37EB,2015-03-17,5,3,3,1,1,3,3,3
5381,PR5839,4500-37EB,2022-03-15,2,4,3,2,0,4,4,3
6500,PR81,4500-37EB,2013-03-17,3,4,5,3,2,2,5,5
6708,PR999,4500-37EB,2016-03-16,4,5,5,3,1,2,3,3


In [170]:
dfPerformance.sort_values(by=['employee_id', 'review_date'], ascending=[True, False], inplace=True)
dfPerformance.drop_duplicates(subset='employee_id', keep='first', inplace=True)

display(f'AFTER: Duplicate performance records for {dup_perf_employee_id[0]}:', dfPerformance[dfPerformance.employee_id.isin(dup_perf_employee_id)])

'AFTER: Duplicate performance records for 4500-37EB:'

,performance_id,employee_id,review_date,environment_satisfaction,job_satisfaction,relationship_satisfaction,training_opportunities_within_year,training_opportunities_taken,work_life_balance,self_rating,manager_rating
5381,PR5839,4500-37EB,2022-03-15,2,4,3,2,0,4,4,3


In [171]:
display(f'BEFORE merge: {dfEmployee.shape = }')

dfEmployee = pd.merge(dfEmployee, dfPerformance, how='left', on='employee_id').set_index('employee_id')

display(f'AFTER merge: {dfEmployee.shape = }.  Note: index set to `employee_id`')
dfEmployee.loc[dup_perf_employee_id].T

'BEFORE merge: dfEmployee.shape = (1470, 23)'

'AFTER merge: dfEmployee.shape = (1470, 32).  Note: index set to `employee_id`'

employee_id,4500-37EB
first_name,Haydon
last_name,Bastable
gender,Male
age,30
business_travel,Frequent Traveller
department,Sales
distance_from_home_km,40
state,CA
ethnicity,Mixed or multiple ethnic groups
education,2


Observations:

1.  Merge and re-indexing successful
1.  **ISSUE WITH DATA**: 10yrs between higher and most-recent performance review, yet `dfEmployee.years_at_company = 1` doesn't reflect:
  - possible issue in HR system, or break in data maintenance
  - [x] generate further `years_*` features -> model / feature importance will show significance
  - if generated features found worthwhile -> correct source data / establish data pipeline / responsibility for caller / API

### Merge look-up values

In [172]:
dfEmployee['education_level'] = dfEmployee.education.apply(lambda id: get_lookup_value(id, dfEducation))

dfEmployee['environment_satisfaction_level'] = dfEmployee.environment_satisfaction.apply(lambda id: get_lookup_value(id, dfSatisfied))
dfEmployee['job_satisfaction_level'] = dfEmployee.job_satisfaction.apply(lambda id: get_lookup_value(id, dfSatisfied))
dfEmployee['relationship_satisfaction_level'] = dfEmployee.relationship_satisfaction.apply(lambda id: get_lookup_value(id, dfSatisfied))
dfEmployee['work_life_balance_level'] = dfEmployee.work_life_balance.apply(lambda id: get_lookup_value(id, dfSatisfied))

dfEmployee['self_rating_level'] = dfEmployee.self_rating.apply(lambda id: get_lookup_value(id, dfRating))
dfEmployee['manager_rating_level'] = dfEmployee.manager_rating.apply(lambda id: get_lookup_value(id, dfRating))

dfEmployee.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1470 entries, 3012-1A41 to 84D4-D4C3
Data columns (total 39 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   first_name                          1470 non-null   object        
 1   last_name                           1470 non-null   object        
 2   gender                              1470 non-null   object        
 3   age                                 1470 non-null   int64         
 4   business_travel                     1470 non-null   object        
 5   department                          1470 non-null   object        
 6   distance_from_home_km               1470 non-null   int64         
 7   state                               1470 non-null   object        
 8   ethnicity                           1470 non-null   object        
 9   education                           1470 non-null   int64         
 10  education_field 

In [173]:
dfEmployee.isnull().sum()

first_name                              0
last_name                               0
gender                                  0
age                                     0
business_travel                         0
department                              0
distance_from_home_km                   0
state                                   0
ethnicity                               0
education                               0
education_field                         0
job_role                                0
marital_status                          0
salary                                  0
stock_option_level                      0
over_time                               0
hire_date                               0
attrition                               0
years_at_company                        0
years_in_most_recent_role               0
years_since_last_promotion              0
years_with_curr_manager                 0
performance_id                        190
review_date                       

## Feature Engineering

In [174]:
dfEmployee.head().T

employee_id,3012-1A41,CBCB-9C9D,95D7-1CE9,47A0-559B,42CC-040A
first_name,Leonelle,Leonerd,Ahmed,Ermentrude,Stace
last_name,Simco,Aland,Sykes,Berrie,Savege
gender,Female,Male,Male,Non-Binary,Female
age,30,38,43,39,29
business_travel,Some Travel,Some Travel,Some Travel,Some Travel,Some Travel
department,Sales,Sales,Human Resources,Technology,Human Resources
distance_from_home_km,27,23,29,12,29
state,IL,CA,CA,IL,CA
ethnicity,White,White,Asian or Asian American,White,White
education,5,4,4,3,2


### Scan for non-standard values (eg capitalisation or spelling mismatches -> consolidate values)

In [175]:
display( dfEmployee.gender.value_counts() )
display( dfEmployee.business_travel.value_counts() )
display( dfEmployee.department.value_counts() )
display( dfEmployee.state.value_counts() )
display( dfEmployee.ethnicity.value_counts() )
display( dfEmployee.education_field.value_counts() )  # TODO Fix 'Marketing'
display( dfEmployee.job_role.value_counts() )
display( dfEmployee.marital_status.value_counts() )
display( dfEmployee.stock_option_level.value_counts() )
display( dfEmployee.over_time.value_counts() )
display( dfEmployee.attrition.value_counts() )
display( dfEmployee.education_level.value_counts() )
display( dfEmployee.environment_satisfaction_level.value_counts() )
display( dfEmployee.job_satisfaction_level.value_counts() )
display( dfEmployee.relationship_satisfaction_level.value_counts() )
display( dfEmployee.work_life_balance_level.value_counts() )
display( dfEmployee.self_rating_level.value_counts() )
display( dfEmployee.manager_rating_level.value_counts() )

gender
Female               675
Male                 651
Non-Binary           124
Prefer Not To Say     20
Name: count, dtype: int64

business_travel
Some Travel           1043
Frequent Traveller     277
No Travel              150
Name: count, dtype: int64

department
Technology         961
Sales              446
Human Resources     63
Name: count, dtype: int64

state
CA    875
NY    419
IL    176
Name: count, dtype: int64

ethnicity
White                               860
Black or African American           207
Mixed or multiple ethnic groups     198
Asian or Asian American             113
American Indian or Alaska Native     50
Native Hawaiian                      26
Other                                16
Name: count, dtype: int64

...

job_satisfaction_level
Neutral              337
Satisfied            336
Dissatisfied         300
Very Satisfied       261
Not Answered         190
Very Dissatisfied     46
Name: count, dtype: int64

relationship_satisfaction_level
Satisfied            332
Dissatisfied         330
Neutral              299
Very Satisfied       269
Not Answered         190
Very Dissatisfied     50
Name: count, dtype: int64

work_life_balance_level
Satisfied            345
Neutral              311
Dissatisfied         305
Very Satisfied       277
Not Answered         190
Very Dissatisfied     42
Name: count, dtype: int64

self_rating_level
Meets Expectation       448
Exceeds Expectation     444
Above and Beyond        388
Not Answered            190
Name: count, dtype: int64

manager_rating_level
Meets Expectation       430
Exceeds Expectation     407
Needs Improvement       240
Above and Beyond        203
Not Answered            190
Name: count, dtype: int64

In [176]:
# Fix 'Marketing'
dfEmployee.education_field = dfEmployee.education_field.apply(lambda f: 'Marketing' if 'Marketing' in f else f)

display( dfEmployee.education_field.value_counts() )

education_field
Computer Science       440
Information Systems    363
Marketing              325
Economics              101
Business Studies        94
Other                   82
Technical Degree        38
Human Resources         27
Name: count, dtype: int64

### Convert categorical features

In [177]:
for col in ['over_time', 'attrition']:
    dfEmployee[col] = dfEmployee[col].apply(lambda val: 0 if 'No' in val else 1)


for col in ['gender', 'business_travel', 'department', 'state', 'ethnicity', 'education_field', 'job_role', 'marital_status', 'stock_option_level', 'education_level',
            'environment_satisfaction_level', 'job_satisfaction_level', 'relationship_satisfaction_level', 'work_life_balance_level', 'self_rating_level', 'manager_rating_level']:
    dfEmployee[col] = dfEmployee[col].astype('category')
    
dfEmployee.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1470 entries, 3012-1A41 to 84D4-D4C3
Data columns (total 39 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   first_name                          1470 non-null   object        
 1   last_name                           1470 non-null   object        
 2   gender                              1470 non-null   category      
 3   age                                 1470 non-null   int64         
 4   business_travel                     1470 non-null   category      
 5   department                          1470 non-null   category      
 6   distance_from_home_km               1470 non-null   int64         
 7   state                               1470 non-null   category      
 8   ethnicity                           1470 non-null   category      
 9   education                           1470 non-null   int64         
 10  education_field 

### Generate date features

In [178]:
dfEmployee.hire_date = pd.to_datetime(dfEmployee.hire_date)

for col in ['hire_date', 'review_date']:
    dfEmployee[col + '_ym'] = dfEmployee[col].dt.to_period('M') # f'{ dfEmployee[col].dt.year }-{ dfEmployee[col].dt.month }'

In [179]:
dfEmployee['generated_years_since_hire'] = \
    (dfEmployee.review_date.dt.year - dfEmployee.hire_date.dt.year) \
        .fillna(0).apply(lambda y: int(y) if 0 <= y else 0)
        
dfEmployee.generated_years_since_hire.describe()

# TODO plot `generated_years` vs `years_at_company`

count    1470.000000
mean        4.692517
std         3.417113
min         0.000000
25%         2.000000
50%         5.000000
75%         8.000000
max        10.000000
Name: generated_years_since_hire, dtype: float64

### Refine feature data types (-> performance)

In [180]:
display("BEFORE: ", dfEmployee[scalar_features(dfEmployee)].info())

dfEmployee.salary = dfEmployee.salary.astype('UInt32')

for col in dfEmployee.select_dtypes(include=[int, float]).columns:
    dfEmployee[col] = dfEmployee[col].round().astype('UInt8')

display("AFTER: ", dfEmployee[scalar_features(dfEmployee)].info())

<class 'pandas.core.frame.DataFrame'>
Index: 1470 entries, 3012-1A41 to 84D4-D4C3
Data columns (total 23 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   age                                 1470 non-null   int64         
 1   distance_from_home_km               1470 non-null   int64         
 2   education                           1470 non-null   int64         
 3   salary                              1470 non-null   int64         
 4   over_time                           1470 non-null   int64         
 5   hire_date                           1470 non-null   datetime64[ns]
 6   attrition                           1470 non-null   int64         
 7   years_at_company                    1470 non-null   int64         
 8   years_in_most_recent_role           1470 non-null   int64         
 9   years_since_last_promotion          1470 non-null   int64         
 10  years_with_curr_

'BEFORE: '

None

<class 'pandas.core.frame.DataFrame'>
Index: 1470 entries, 3012-1A41 to 84D4-D4C3
Data columns (total 23 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   age                                 1470 non-null   UInt8         
 1   distance_from_home_km               1470 non-null   UInt8         
 2   education                           1470 non-null   UInt8         
 3   salary                              1470 non-null   UInt32        
 4   over_time                           1470 non-null   UInt8         
 5   hire_date                           1470 non-null   datetime64[ns]
 6   attrition                           1470 non-null   UInt8         
 7   years_at_company                    1470 non-null   UInt8         
 8   years_in_most_recent_role           1470 non-null   UInt8         
 9   years_since_last_promotion          1470 non-null   UInt8         
 10  years_with_curr_

'AFTER: '

None

## Exploratory Data Analysis (EDA)

In [181]:
dfEmployee.head().T

employee_id,3012-1A41,CBCB-9C9D,95D7-1CE9,47A0-559B,42CC-040A
first_name,Leonelle,Leonerd,Ahmed,Ermentrude,Stace
last_name,Simco,Aland,Sykes,Berrie,Savege
gender,Female,Male,Male,Non-Binary,Female
age,30,38,43,39,29
business_travel,Some Travel,Some Travel,Some Travel,Some Travel,Some Travel
department,Sales,Sales,Human Resources,Technology,Human Resources
distance_from_home_km,27,23,29,12,29
state,IL,CA,CA,IL,CA
ethnicity,White,White,Asian or Asian American,White,White
education,5,4,4,3,2


### Drop features and NaN -> 0 (see homework instructions)

In [182]:
# del df['student_id']
# df.fillna(0, inplace=True)
        
# df.head().T

### Split the data

#### Training, Testing, Validation, & Full (Training + Validation)

In [183]:
# df_val, df_test, df_train, df_full = validation_testing_training_full_split(df, seed=1)

# nTotal = len(df)
# nVal = len(df_val)
# nTest = len(df_test)
# nTrain = len(df_train)
# nFull = len(df_full)

# round(nVal/nTotal, 1), round(nTest/nTotal, 1), round(nTrain/nTotal, 1), round(nFull/nTotal,1), round(nTotal/nTotal)

#### Split out `y` (target feature) from all datasets 

In [184]:
# df_val, y_val = y_split(df_val, yColumn='jamb_score')
# df_test, y_test = y_split(df_test, yColumn='jamb_score')
# df_train, y_train = y_split(df_train, yColumn='jamb_score')
# df_full, y_full = y_split(df_full, yColumn='jamb_score')

# assert df_val.shape[1] == df_test.shape[1] and df_test.shape[1] == df_train.shape[1] and df_train.shape[1] == df_full.shape[1]
# assert len(y_val) == df_val.shape[0] and len(y_test) == df_test.shape[0] and len(y_train) == df_train.shape[0] and len(y_full) == df_full.shape[0]

In [185]:
# df_train.head().T

## Modelling

### Drop superfluous data frames & features (-> performace)

In [186]:
del dfEducation
del dfRating
del dfSatisfied

del dfPerformance

In [187]:
del dfEmployee['first_name']
del dfEmployee['last_name']
# del dfEmployee['gender']
del dfEmployee['education']
del dfEmployee['hire_date']
del dfEmployee['performance_id']
del dfEmployee['review_date']

del dfEmployee['environment_satisfaction']
del dfEmployee['job_satisfaction']
del dfEmployee['relationship_satisfaction']
del dfEmployee['work_life_balance']

del dfEmployee['self_rating']
del dfEmployee['manager_rating']

display('scalar_features:', dfEmployee[scalar_features(dfEmployee)].info())
display('categorical_features:', dfEmployee[categorical_features(dfEmployee)].info())

<class 'pandas.core.frame.DataFrame'>
Index: 1470 entries, 3012-1A41 to 84D4-D4C3
Data columns (total 14 columns):
 #   Column                              Non-Null Count  Dtype    
---  ------                              --------------  -----    
 0   age                                 1470 non-null   UInt8    
 1   distance_from_home_km               1470 non-null   UInt8    
 2   salary                              1470 non-null   UInt32   
 3   over_time                           1470 non-null   UInt8    
 4   attrition                           1470 non-null   UInt8    
 5   years_at_company                    1470 non-null   UInt8    
 6   years_in_most_recent_role           1470 non-null   UInt8    
 7   years_since_last_promotion          1470 non-null   UInt8    
 8   years_with_curr_manager             1470 non-null   UInt8    
 9   training_opportunities_within_year  1280 non-null   UInt8    
 10  training_opportunities_taken        1280 non-null   UInt8    
 11  hire_date

'scalar_features:'

None

<class 'pandas.core.frame.DataFrame'>
Index: 1470 entries, 3012-1A41 to 84D4-D4C3
Data columns (total 16 columns):
 #   Column                           Non-Null Count  Dtype   
---  ------                           --------------  -----   
 0   gender                           1470 non-null   category
 1   business_travel                  1470 non-null   category
 2   department                       1470 non-null   category
 3   state                            1470 non-null   category
 4   ethnicity                        1470 non-null   category
 5   education_field                  1470 non-null   category
 6   job_role                         1470 non-null   category
 7   marital_status                   1470 non-null   category
 8   stock_option_level               1470 non-null   category
 9   education_level                  1470 non-null   category
 10  environment_satisfaction_level   1470 non-null   category
 11  job_satisfaction_level           1470 non-null   category
 12

'categorical_features:'

None

In [192]:
dfEmployee.head().T

employee_id,3012-1A41,CBCB-9C9D,95D7-1CE9,47A0-559B,42CC-040A
gender,Female,Male,Male,Non-Binary,Female
age,30,38,43,39,29
business_travel,Some Travel,Some Travel,Some Travel,Some Travel,Some Travel
department,Sales,Sales,Human Resources,Technology,Human Resources
distance_from_home_km,27,23,29,12,29
state,IL,CA,CA,IL,CA
ethnicity,White,White,Asian or Asian American,White,White
education_field,Marketing,Marketing,Marketing,Computer Science,Technical Degree
job_role,Sales Executive,Sales Executive,HR Business Partner,Engineering Manager,Recruiter
marital_status,Divorced,Single,Married,Married,Single


## Question 1: Training the model

In [ ]:
# model, dv = fit(
#     model=DecisionTreeRegressor(max_depth=1), 
#     dv=DictVectorizer(sparse=True),
#     df=df_train, 
#     y=y_train,
# )
# model, dv

In [ ]:
# print(export_text(model, feature_names=dv.feature_names_))

Which feature is used for splitting the data?

* `study_hours_per_week`  `<--`
* `attendance_rate`
* `teacher_quality`
* `distance_to_school`

## Question 2: Random Forest

In [ ]:
# model, dv = fit(
#     model=RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1), 
#     dv=DictVectorizer(sparse=True),
#     df=df_train, 
#     y=y_train,
# )
# model, dv

In [ ]:
# y_val_pred = decide(model, dv, df_val)
# root_mean_squared_error(y_val, y_val_pred)

What's the RMSE of this model on validation?

* 22.13
* 42.13 `<--`
* 62.13
* 82.12

## Question 3: Random Forest Tuning (n_estimators)

In [ ]:
# scores = []
# for estimators in tqdm( range(10, 201, 10) ):
#     model, dv = fit(
#         model=RandomForestRegressor(n_estimators=estimators, random_state=1, n_jobs=-1),
#         dv=DictVectorizer(sparse=True),
#         df=df_train,
#         y=y_train,
#     )
    
#     y_val_pred = decide(model, dv, df_val)
#     scores.append((estimators, root_mean_squared_error(y_val, y_val_pred)))
    
# df_scores = pd.DataFrame(scores, columns=['n_estimators', 'rmse'])
# df_scores

After which value of `n_estimators` does RMSE stop improving?
Consider 3 decimal places for calculating the answer.

- 10
- 25
- 80  `<--`
- 200

## Question 4: Random Forest Tuning (n_estimators & max_depth)

In [ ]:
# scores = []
# for depth in [10, 15, 20, 25]:
#     for estimators in tqdm( range(10, 201, 10), desc=f'max_depth = {depth}' ):
#         model, dv = fit(
#             model=RandomForestRegressor(max_depth=depth, n_estimators=estimators, random_state=1, n_jobs=-1),
#             dv=DictVectorizer(sparse=True),
#             df=df_train,
#             y=y_train,
#         )
        
#         y_val_pred = decide(model, dv, df_val)
#         scores.append((depth, estimators, root_mean_squared_error(y_val, y_val_pred)))
    
# df_scores = pd.DataFrame(scores, columns=['max_depth', 'n_estimators', 'rmse'])
# df_scores.groupby(by='max_depth').mean()

What's the best `max_depth`, using the mean RMSE?

* 10 `<--`
* 15
* 20
* 25

## Question 5: Feature Imporance

In [ ]:
# model, dv = fit(
#     model=RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1),
#     dv=DictVectorizer(sparse=True),
#     df=df_train,
#     y=y_train,
# )

# df_scores = pd.DataFrame(zip(dv.feature_names_, model.feature_importances_), columns=['feature', 'importance'])
# df_scores.sort_values(by='importance', ascending=False)

What's the most important feature (among these 4)? 

* `study_hours_per_week` `<--`
* `attendance_rate`
* `distance_to_school`
* `teacher_quality`

## Question 6: Gradient Boosting

In [ ]:
# for eta in [0.3, 0.1]:

#     xgb_params = {
#         'eta': eta, 
#         'max_depth': 6,
#         'min_child_weight': 1,
        
#         'objective': 'reg:squarederror',
#         'nthread': 2,
        
#         'seed': 1,
#         'verbosity': 1,
#     }

#     X_train, dv = one_hot_encode(df_train, dv=DictVectorizer(sparse=True), fit=True)
#     X_val, _ = one_hot_encode(df_val, dv=dv)

#     dX_train = xgb.DMatrix(X_train, label=y_train, feature_names=dv.feature_names_)
#     dX_val = xgb.DMatrix(X_val, label=y_val, feature_names=dv.feature_names_)

#     model = xgb.train(xgb_params, dX_train, num_boost_round=100)

#     y_val_pred = model.predict(dX_val)
    
#     display(f'{eta=} -> rmse={root_mean_squared_error(y_val, y_val_pred):.4f}')

Which eta leads to the best RMSE score on the validation dataset?

* 0.3
* 0.1 `<--`
* Both give equal value